# NorthStar Urban Mobility and Logistics
## SQL in R Analysis

## 1. Install and Load sqldf

In [4]:
install.packages("sqldf")
library(sqldf)
print("sqldf ready")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘gsubfn’, ‘proto’, ‘RSQLite’, ‘chron’


Loading required package: gsubfn

Loading required package: proto

Warning message:
“no DISPLAY variable so Tk is not available”
Loading required package: RSQLite



[1] "sqldf ready"


## 2. Load the Data

In [5]:
base_url <- "https://raw.githubusercontent.com/chelsajn/DBAnew/main/"

orders     <- read.csv(paste0(base_url, "orders.csv"))
deliveries <- read.csv(paste0(base_url, "deliveries.csv"))
customers  <- read.csv(paste0(base_url, "customers.csv"))
drivers    <- read.csv(paste0(base_url, "drivers.csv"))
vehicles   <- read.csv(paste0(base_url, "vehicles.csv"))
hubs       <- read.csv(paste0(base_url, "hubs.csv"))
complaints <- read.csv(paste0(base_url, "complaints.csv"))
incidents  <- read.csv(paste0(base_url, "incidents.csv"))
app_events <- read.csv(paste0(base_url, "app_events.csv"))

## 3. Explore the Data

In [6]:
#orders table
head(orders)

,order_id,customer_id,service_type,order_created_at,promised_window_hours,pickup_zone,dropoff_zone,priority_level,order_value,booking_channel,special_handling_flag
,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<dbl>,<chr>,<int>
1,O00001,C0292,Passenger,2024-08-20 14:43:00,6,Airport,South,Medium,126.65,App,0
2,O00002,C0459,Passenger,2024-05-14 22:16:00,24,North,AIRPORT,Low,109.30,App,0
3,O00003,C0161,Passenger,2025-09-02 14:37:00,4,West,AIRPORT,High,33.50,Phone,0
4,O00004,C0520,Parcel,2025-01-11 17:15:00,2,RiverSide,North,Medium,10.04,App,1
5,O00005,C0558,Retail,2025-02-17 19:32:00,12,Riverside,SOUTH,Low,125.58,Phone,0
6,O00006,C0437,Retail,2024-08-05 04:55:00,1,CENTRAL,East,High,151.44,Web,1


In [7]:
#the deliveries table
head(deliveries)

,delivery_id,order_id,driver_id,vehicle_id,hub_id,dispatch_time,delivery_completed_at,delivery_status,route_distance_km,manual_route_override_count,proof_of_completion_missing,customer_rating_post_delivery,fuel_or_charge_cost
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<int>,<int>,<dbl>,<dbl>
1,DL00001,O00938,D004,V056,H05,2024-06-18 10:57:00,2024-06-19 09:05:59.904311,Failed,17.26,1,0,3.07,12.05
2,DL00002,O00004,D138,V007,H02,2025-01-11 18:45:00,2025-01-11 17:39:00.000000,OnTime,10.34,1,0,5.00,13.41
3,DL00003,O00639,D006,V049,H02,2025-06-02 20:39:00,2025-06-02 21:45:32.366770,OnTime,7.92,0,0,4.98,8.51
4,DL00004,O00313,D116,V055,H02,2024-03-08 23:31:00,2024-03-09 23:30:08.103702,Delayed,16.42,0,0,4.18,13.62
5,DL00005,O00844,D108,V034,H01,2025-09-21 11:43:00,2025-09-21 15:45:34.131056,OnTime,14.52,1,0,4.18,9.22
6,DL00006,O00029,D037,V098,H03,2024-09-11 12:40:00,2024-09-12 17:11:52.384869,Delayed,13.84,0,0,1.57,9.58


## 4. Data Cleaning

In [8]:

#fixing inconsistent zone names

orders$pickup_zone[orders$pickup_zone %in% c("north", "NORTH")] <- "North"
orders$dropoff_zone[orders$dropoff_zone %in% c("south", "SOUTH")] <- "South"

customers$home_zone[customers$home_zone %in% c("east", "EAST")] <- "East"

drivers$base_zone[drivers$base_zone %in% c("west", "WEST")] <- "West"

vehicles$assigned_zone[vehicles$assigned_zone %in% c("central", "CENTRAL", "Ctr")] <- "Central"

cat("zones after cleaning in orders:\n")
print(unique(orders$pickup_zone))
cat("\nzones after cleaning in drivers:\n")
print(unique(drivers$base_zone))

zones after cleaning in orders:
 [1] "Airport"   "North"     "West"      "RiverSide" "Riverside" "CENTRAL"  
 [7] "South"     "WEST"      "East"      "Ctr"       "SOUTH"     "Central"  
[13] "AIRPORT"   "EAST"     

zones after cleaning in drivers:
 [1] "AIRPORT"   "Central"   "Airport"   "north"     "CENTRAL"   "North"    
 [7] "SOUTH"     "West"      "East"      "NORTH"     "EAST"      "Riverside"
[13] "South"     "RiverSide" "Ctr"      


In [9]:
#filling missing ratings with the median so we don't lose those rows
med <- median(deliveries$customer_rating_post_delivery, na.rm = TRUE)
deliveries$customer_rating_post_delivery[is.na(deliveries$customer_rating_post_delivery)] <- med
cat("missing ratings remaining:", sum(is.na(deliveries$customer_rating_post_delivery)), "\n")

missing ratings remaining: 0 


## 5. SQL Queries Using sqldf

In [10]:
#parcel orders
q1 <- sqldf("SELECT order_id, service_type, pickup_zone, order_value
             FROM orders
             WHERE service_type = 'Parcel'")
cat("parcel orders found:", nrow(q1), "\n")
print(head(q1))

parcel orders found: 308 
  order_id service_type pickup_zone order_value
1   O00004       Parcel   RiverSide       10.04
2   O00008       Parcel   Riverside       35.06
3   O00011       Parcel        WEST       79.10
4   O00016       Parcel     Central       92.66
5   O00017       Parcel     AIRPORT       60.90
6   O00026       Parcel        East       51.09


In [11]:
#checking which orders are both high priority AND worth more than 100
q2 <- sqldf("SELECT order_id, service_type, priority_level, order_value
             FROM orders
             WHERE priority_level = 'High' AND order_value > 100")
cat("high priority orders over 100:", nrow(q2), "\n")
print(head(q2))

high priority orders over 100: 123 
  order_id service_type priority_level order_value
1   O00006       Retail           High      151.44
2   O00015     Business           High      145.58
3   O00018    Passenger           High      121.11
4   O00024    Passenger           High      132.20
5   O00056       Parcel           High      160.35
6   O00061    Passenger           High      238.90


In [12]:
#sorting deliveries by fuel cost to see which ones cost the most
q3 <- sqldf("SELECT delivery_id, route_distance_km, fuel_or_charge_cost
             FROM deliveries
             ORDER BY fuel_or_charge_cost DESC")
cat("top 10 most expensive deliveries:\n")
print(head(q3, 10))

top 10 most expensive deliveries:
   delivery_id route_distance_km fuel_or_charge_cost
1      DL00897             33.64               29.43
2      DL00144             25.98               27.38
3      DL00713             21.40               26.99
4      DL00664             20.64               25.46
5      DL00119             32.37               25.09
6      DL00052             26.38               24.54
7      DL00287             36.08               24.50
8      DL00806             40.11               24.27
9      DL00090             33.86               24.20
10     DL00373             11.36               23.60


In [13]:
#checking the deliveries that have a rating
q4 <- sqldf("SELECT delivery_id, delivery_status, customer_rating_post_delivery
             FROM deliveries
             WHERE customer_rating_post_delivery IS NOT NULL")
cat("deliveries with a rating:", nrow(q4), "\n")
print(head(q4))

deliveries with a rating: 950 
  delivery_id delivery_status customer_rating_post_delivery
1     DL00001          Failed                          3.07
2     DL00002          OnTime                          5.00
3     DL00003          OnTime                          4.98
4     DL00004         Delayed                          4.18
5     DL00005          OnTime                          4.18
6     DL00006         Delayed                          1.57


In [14]:
#grouping by status to see how many deliveries failed, and what the average rating and fuel cost is per group
q5 <- sqldf("SELECT delivery_status,
                    COUNT(*) AS total,
                    ROUND(AVG(customer_rating_post_delivery), 2) AS avg_rating,
                    ROUND(AVG(fuel_or_charge_cost), 2) AS avg_fuel_cost
             FROM deliveries
             GROUP BY delivery_status")
cat("Summary by delivery status:\n")
print(q5)

Summary by delivery status:
  delivery_status total avg_rating avg_fuel_cost
1         Delayed   202       3.14         13.14
2          Failed   132       3.06         13.15
3          OnTime   616       4.28         12.68


In [15]:
#only showing zones with more than 50 orders so we focus on the busiest ones
q6 <- sqldf("SELECT pickup_zone,
                    COUNT(*) AS total_orders,
                    ROUND(AVG(order_value), 2) AS avg_order_value
             FROM orders
             GROUP BY pickup_zone
             HAVING COUNT(*) > 50
             ORDER BY total_orders DESC")
cat("busy zones (over 50 orders):\n")
print(q6)

busy zones (over 50 orders):
   pickup_zone total_orders avg_order_value
1        North          174           91.03
2         East          104           92.22
3        South          103           92.40
4         EAST          103           91.33
5    RiverSide           86           80.38
6      Airport           85          108.85
7         WEST           84           89.20
8          Ctr           80           94.50
9      Central           79           77.20
10     CENTRAL           79           93.58
11       SOUTH           78           88.18
12        West           71           87.18
13   Riverside           65           91.90
14     AIRPORT           59           96.74


In [16]:
#joining orders and deliveries together to see both in one table
q7 <- sqldf("SELECT o.order_id, o.service_type, o.pickup_zone,
                    d.delivery_status, d.customer_rating_post_delivery
             FROM orders o
             JOIN deliveries d ON o.order_id = d.order_id")
cat("joined table has", nrow(q7), "rows\n")
print(head(q7))

joined table has 950 rows
  order_id service_type pickup_zone delivery_status
1   O00001    Passenger     Airport          OnTime
2   O00003    Passenger        West         Delayed
3   O00004       Parcel   RiverSide          OnTime
4   O00005       Retail   Riverside          OnTime
5   O00007     Business     CENTRAL         Delayed
6   O00008       Parcel   Riverside          OnTime
  customer_rating_post_delivery
1                          4.29
2                          3.70
3                          5.00
4                          4.38
5                          3.93
6                          5.00


In [17]:
#looking at just the failed deliveries and sorting by how many route overrides they had
#to check if drivers overriding routes is linked to failures
q8 <- sqldf("SELECT o.order_id, o.service_type, o.pickup_zone,
                    d.delivery_status, d.manual_route_override_count
             FROM orders o
             JOIN deliveries d ON o.order_id = d.order_id
             WHERE d.delivery_status = 'Failed'
             ORDER BY d.manual_route_override_count DESC")
cat("failed deliveries sorted by route overrides:\n")
print(head(q8, 10))

failed deliveries sorted by route overrides:
   order_id service_type pickup_zone delivery_status
1    O00510      Medical        WEST          Failed
2    O00618       Parcel     CENTRAL          Failed
3    O00211     Business     Airport          Failed
4    O01207     Business     CENTRAL          Failed
5    O00750       Retail       North          Failed
6    O00660    Passenger     Central          Failed
7    O01011     Business     AIRPORT          Failed
8    O00833       Parcel     Central          Failed
9    O00619       Retail     CENTRAL          Failed
10   O00934     Business       North          Failed
   manual_route_override_count
1                            4
2                            4
3                            4
4                            3
5                            3
6                            3
7                            3
8                            3
9                            3
10                           3


In [18]:
#calculating the failure rate for each service type as a percentage
q9 <- sqldf("SELECT o.service_type,
                    COUNT(*) AS total,
                    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed,
                    ROUND(SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS failure_rate_pct
             FROM orders o
             JOIN deliveries d ON o.order_id = d.order_id
             GROUP BY o.service_type
             ORDER BY failure_rate_pct DESC")
cat("Failure rate by service type:\n")
print(q9)

Failure rate by service type:
  service_type total failed failure_rate_pct
1     Business   126     25             19.8
2      Medical   108     16             14.8
3    Passenger   262     38             14.5
4       Retail   224     28             12.5
5       Parcel   230     25             10.9


In [19]:
#joining three tables at once to see how driver type affects customer satisfaction
q10 <- sqldf("SELECT o.service_type,
                     dr.employment_type,
                     ROUND(AVG(dr.driver_rating), 2) AS avg_driver_rating,
                     ROUND(AVG(d.customer_rating_post_delivery), 2) AS avg_customer_rating,
                     COUNT(*) AS total
              FROM orders o
              JOIN deliveries d ON o.order_id = d.order_id
              JOIN drivers dr ON d.driver_id = dr.driver_id
              GROUP BY o.service_type, dr.employment_type
              ORDER BY avg_customer_rating DESC")
cat("Driver type vs customer rating by service:\n")
print(q10)

Driver type vs customer rating by service:
   service_type employment_type avg_driver_rating avg_customer_rating total
1      Business        Contract              4.23                4.11    18
2        Parcel        PartTime              4.15                4.05    63
3        Retail        Contract              4.12                4.03    34
4       Medical        FullTime              4.18                3.94    72
5      Business        PartTime              4.30                3.93    42
6     Passenger        Contract              4.02                3.92    29
7        Retail        FullTime              4.20                3.92   131
8        Parcel        FullTime              4.16                3.91   140
9     Passenger        PartTime              4.22                3.90    60
10    Passenger        FullTime              4.12                3.82   173
11      Medical        PartTime              4.24                3.76    18
12     Business        FullTime              

In [28]:
#INSERT - adding a new test order row
new_order <- data.frame(
  order_id = "ORD9999",
  customer_id = "CUST999",
  service_type = "Parcel",
  order_created_at = "2024-01-01",
  promised_window_hours = 24,
  pickup_zone = "North",
  dropoff_zone = "South",
  priority_level = "High",
  order_value = 85.00,
  booking_channel = "App",
  special_handling_flag = 0,
  stringsAsFactors = FALSE
)

orders <- rbind(orders, new_order)
cat("Row inserted. New row count:", nrow(orders), "\n")

#UPDATE - change priority of test order
orders$priority_level[orders$order_id == "ORD9999"] <- "Low"
cat("Updated priority:", orders$priority_level[orders$order_id == "ORD9999"], "\n")

#DELETE - remove the test order
orders <- orders[orders$order_id != "ORD9999", ]
cat("Row deleted. Row count after deletion:", nrow(orders), "\n")

Row inserted. New row count: 1251 
Updated priority: Low 
Row deleted. Row count after deletion: 1250 


## Summary

- This notebook demonstrated the use of SQL within R using the sqldf package to query
the NorthStar dataset. Zone names were standardised across all four relevant dataframes
before querying to ensure consistent results.

- Ten SQL queries were executed covering SELECT, WHERE, ORDER BY, GROUP BY, HAVING,
IS NOT NULL, aggregate functions, and JOIN operations across two and three tables.

- Key findings include a clear link between manual route overrides and delivery failure,
and lower customer satisfaction scores among contract drivers compared to permanent staff.

- INSERT, UPDATE, and DELETE operations were also performed on the orders dataframe,
confirming full data manipulation capability within R beyond querying alone.
